In [2]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re
from rapidfuzz import process, fuzz

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

from scripts.text_matching import normalise_text,test_similarity


In [3]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
unmatched_file = Path(project_root / "data/people/unmatched.json")

In [4]:
with open(people_file , "r") as f:
   people = json.load(f)
with open(variants_file , "r") as f:
   variants = json.load(f)
with open(unmatched_file, "r") as f:
   unmatched = json.load(f)

rprint(dict(list(unmatched.items())[:2]))

{
    'rommel, otto': [
        {
            'display_name': 'ROMMEL, Otto',
            'composite_id': 'wien_210_9_10',
            'sort_order': 1,
            'is_author': True,
            'is_editor': False,
            'is_contributor': False,
            'is_translator': False
        },
        {
            'display_name': 'Rommel, Otto',
            'composite_id': 'briefe_45_2_15',
            'sort_order': 2,
            'is_author': False,
            'is_editor': True,
            'is_contributor': False,
            'is_translator': False
        }
    ],
    'stigler-fuchs, margarete von': [
        {
            'display_name': 'STIGLER-FUCHS, Margarete von',
            'composite_id': 'wien_220_9_10',
            'sort_order': 1,
            'is_author': True,
            'is_editor': False,
            'is_contributor': False,
            'is_translator': False
        },
        {
            'display_name': 'STIGLER-FUCHS, Margarete von',
            'composite_id': 'wien_221_9_10',
            'sort_order': 1,
            'is_author': True,
            'is_editor': False,
            'is_contributor': False,
            'is_translator': False
        }
    ]
}

In [5]:
variants_dict = {}
for variant in variants:
    variants_dict[normalise_text(variant["variant_normalised"])] = variant["person_id"]

choices = list(variants_dict.keys())
# rprint(dict(list(variants_dict.items())[:5]))
rprint(choices[:3])

unmatched_list = list(unmatched.keys())
rprint(unmatched_list[:3])

['abel, othenio', 'abel, otto', 'kurt absolon']

['rommel, otto', 'stigler-fuchs, margarete von', 'stadt mödling']

In [6]:
sample = unmatched_list[0]

result = process.extractOne(sample, choices, scorer=fuzz.ratio, score_cutoff=70)
rprint(sample , "→", result)

result_top3 = process.extract(sample, choices, scorer=fuzz.ratio, limit=3, score_cutoff=70)
rprint(f"top3: {result_top3}")


rommel, otto →
('muehl, otto', 78.26086956521739, 5665)

top3: [('muehl, otto', 78.26086956521739, 5665), ('abel, otto', 72.72727272727273, 1), ('merk, otto', 
72.72727272727273, 5440)]

In [7]:
results = {}

for display_norm in unmatched_list:
    top_matches = process.extract(display_norm, choices, scorer=fuzz.ratio, limit=3, score_cutoff=70)
    results[display_norm] = top_matches

rprint(dict(list(results.items())[:5]))

{
    'rommel, otto': [
        ('muehl, otto', 78.26086956521739, 5665),
        ('abel, otto', 72.72727272727273, 1),
        ('merk, otto', 72.72727272727273, 5440)
    ],
    'stigler-fuchs, margarete von': [],
    'stadt mödling': [],
    'renolder, klemens': [('brentano, clemens', 70.58823529411764, 978)],
    'fritz, elisabeth': [
        ('gürt, elisabeth', 83.87096774193549, 2960),
        ('riedt, ruth-elisabeth', 75.67567567567568, 6662),
        ('nemeth, elisabeth', 72.72727272727273, 5809)
    ]
}

In [8]:
high = {}
mid = {}
low = {}
no_match = {}


for display_norm, matches in results.items():
    books = unmatched[display_norm]

    if not matches:
        no_match[display_norm] = books
        continue

    best_score = matches[0][1]
    if best_score >= 95:
        high[display_norm] = matches
    elif best_score >= 85:
        mid[display_norm] = matches
    else:
        low[display_norm] = matches

total = len(results)
t_high = len(high)
t_mid = len(mid)
t_low = len(low)
t_no = len(no_match)

rprint(f"results: \n total: {total}")
rprint(f"high: {t_high}")
rprint(f"mid: {t_mid}")
rprint(f"low: {t_low}")
rprint(f"no: {t_no}")



# with open("likely_matched.json", "w") as f:
#     json.dump(high, f, ensure_ascii=False, indent=2)
# with open("middling_matched.json", "w") as f:
#     json.dump(mid, f, ensure_ascii=False, indent=2)

# with open("low_likelyhood.json", "w") as f:
#     json.dump(low, f, ensure_ascii=False, indent=2)


results: 
 total: 1575

high: 64

mid: 158

low: 772

no: 581